In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [0]:
def read_data(path:str,format:str="parquet")->DataFrame:
    """takes in path and data format and returns spark DataFrame"""

    if format.lower() == "csv":
        return spark.read.format("csv").option("header",True).load(path)

    elif format.lower() == "table":
        return spark.read.table(path)

    elif format.lower() == "parquet":
        return spark.read.parquet(path)

    elif format.lower() == "delta":
        return spark.read.load(path)
    

def write_data(df:DataFrame,path:str,partition_cols:list[str]|str=None)->None:
    """taken in spark Dataframe, path where data has to be stored, 
    partition columns and saves the data to given path in parquet format"""
    if partition_cols==None:
            df.write.format("parquet").mode('overwrite').save(path)
    else:
        df.write.format("parquet").partitionBy(*partition_cols).mode('overwrite').save(path)

In [0]:
def compute_segments(data):
    """
    Compute simple demand segments for each ID.

    Segments:
        - new
        - smooth
        - erratic
        - intermittent
        - lumpy
        - dead
    """

    df = data.copy()

    # -----------------------------
    # Basic series statistics
    # -----------------------------

    stats = (
        df.groupby("id")
        .agg(
            weeks=("yearweek", "nunique"),
            total_sales=("sales", "sum"),
            avg_sales=("sales", "mean"),
            std_sales=("sales", "std"),
            zero_weeks=("sales", lambda x: (x == 0).sum()),
            positive_weeks=("sales", lambda x: (x > 0).sum())
        )
    )

    # -----------------------------
    # Zero share
    # -----------------------------

    stats["zero_share"] = (
        stats["zero_weeks"] / stats["weeks"]
    )

    # -----------------------------
    # ADI
    # Average Demand Interval
    # -----------------------------

    stats["adi"] = np.where(
        stats["positive_weeks"] > 0,
        stats["weeks"] / stats["positive_weeks"],
        np.inf
    )

    # -----------------------------
    # CV²
    # -----------------------------

    stats["cv2"] = (
        stats["std_sales"] /
        stats["avg_sales"].replace(0, np.nan)
    ) ** 2

    stats["cv2"] = stats["cv2"].fillna(np.inf)

    # -----------------------------
    # Demand classification
    # -----------------------------

    stats["segment"] = np.select(
        [
            stats["zero_share"] >= 0.95,

            stats["weeks"] < 52,

            (stats["adi"] < 1.32) &
            (stats["cv2"] < 0.49),

            (stats["adi"] < 1.32) &
            (stats["cv2"] >= 0.49),

            (stats["adi"] >= 1.32) &
            (stats["cv2"] < 0.49),
        ],
        [
            "dead",
            "new",
            "smooth",
            "erratic",
            "intermittent"
        ],
        default="lumpy"
    )
    segments = stats.reset_index()[['id','segment']]


    return data.merge(segments,on=['id'],how='left').drop_duplicates()